In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import pipeline

In [4]:
df = pd.read_csv("../data/clean_hr_data.csv")

In [5]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

In [9]:
# BOW method 
# You fit and transform on the column
# you only transform the y1 and y2
vectorizer_bow = CountVectorizer()
X = vectorizer_bow.fit_transform(df['clean_job_title'])
y1 = vectorizer_bow.transform([measure_rank_1])
y2 = vectorizer_bow.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

df["similarity_1_bow"] = score_1.flatten()
df[df["similarity_1_bow"]>0.3][["clean_job_title", "similarity_1_bow"]].sort_values(
    by="similarity_1_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_1_bow
21,aspiring human resources manager seek internsh...,0.745356
47,aspiring human resources manager | graduating ...,0.615457
5,aspiring human resources specialist greater ne...,0.577350
6,student humber college aspiring human resource...,0.577350
20,business management major aspiring human resou...,0.547723
2,aspiring human resources professional raleigh ...,0.547723
12,aspiring human resources management student se...,0.522233
30,aspiring human resources professional | energe...,0.500000
13,seek human resources opportunities chicago ill...,0.436436
36,human resources management major milpitas cali...,0.436436


In [8]:
df["similarity_2_bow"] = score_2.flatten()
df[df["similarity_2_bow"]>0.3][["clean_job_title", "similarity_2_bow"]].sort_values(
    by="similarity_2_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bow
21,aspiring human resources manager seek internsh...,0.730297
47,aspiring human resources manager | graduating ...,0.603023
13,seek human resources opportunities chicago ill...,0.534522
22,human resources professional greater boston ar...,0.534522
36,human resources management major milpitas cali...,0.534522
37,director human resources ey greater atlanta ...,0.500000
26,human resources generalist schwan amerika birl...,0.500000
15,human resources staffing recruiting profession...,0.471405
46,seek human resources position las vegas nevada...,0.471405
6,student humber college aspiring human resource...,0.471405


In [ ]:
# Tfidf method
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df['clean_job_title'])
y1 = vectorizer_tfidf.transform([measure_rank_1])
y2 = vectorizer_tfidf.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)


df["similarity_1_tfidf"] = score_1.flatten()
df[df["similarity_1_tfidf"]>0.3][["clean_job_title", "similarity_1_tfidf"]].sort_values(
    by="similarity_1_tfidf",
    ascending=False
).head(10)

In [ ]:
df["similarity_2_tfidf"] = score_2.flatten()
df[df["similarity_2_tfidf"]>0.3][["clean_job_title", "similarity_2_tfidf"]].sort_values(
    by="similarity_2_tfidf",
    ascending=False
).head(10)

In [ ]:
#%pip install gensim

model = api.load('glove-wiki-gigaword-100')


# apply has the tendency to work on the rows automatically 
def get_title_vector(row):
    words = row.split()

    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    return np.mean(vectors, axis=0)


    
df['vector_for_modeling'] = df["clean_job_title"].apply(get_title_vector)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)


X_glove = np.vstack(df["vector_for_modeling"])
# to make it work you need to use vstack function
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_glove
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_glove
)

df["similarity_1_glove"] = score_1.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_1_glove"]>0.3][["clean_job_title", "similarity_1_glove"]].sort_values(
    by="similarity_1_glove",
    ascending=False
).head(10)

In [ ]:
df["similarity_2_glove"] = score_2.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_2_glove"]>0.3][["clean_job_title", "similarity_2_glove"]].sort_values(
    by="similarity_2_glove",
    ascending=False
).head(10)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def get_title_vector_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
# pt is py_torch, truncation for fitting what the model allow in size, padding makes the size the same
    with torch.no_grad():
        outputs = bert(**inputs)


    vector = outputs.last_hidden_state.mean(dim=1)

    return vector.squeeze().numpy()
# make the tensor a numpy array with one dimension 
df['bert_for_modeling'] = df["clean_job_title"].apply(get_title_vector_bert)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)



X_bert = np.vstack(df["bert_for_modeling"])

y1_vector = get_title_vector_bert(measure_rank_1)
y2_vector = get_title_vector_bert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bert
)

df["similarity_1_bert"] = score_1.flatten()

df[df["similarity_1_bert"]>0.3][["clean_job_title", "similarity_1_bert"]].sort_values(
    by="similarity_1_bert",
    ascending=False
).head(10)

In [ ]:
df["similarity_2_bert"] = score_2.flatten()

df[df["similarity_2_bert"]>0.3][["clean_job_title", "similarity_2_bert"]].sort_values(
    by="similarity_2_bert",
    ascending=False
).head(10)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_title_vector_sbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = sbert(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Mean pooling, ignoring padding tokens
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["sbert_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_sbert
)


s_bert = np.vstack(df["sbert_for_modeling"])

y1_vector = get_title_vector_sbert(measure_rank_1)
y2_vector = get_title_vector_sbert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    s_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    s_bert
)

df["similarity_1_sbert"] = score_1.flatten()

df[df["similarity_1_sbert"]>0.3][["clean_job_title", "similarity_1_sbert"]].sort_values(
    by="similarity_1_sbert",
    ascending=False
).head(10)

In [ ]:
df["similarity_2_sbert"] = score_2.flatten()

df[df["similarity_2_sbert"]>0.3][["clean_job_title", "similarity_2_sbert"]].sort_values(
    by="similarity_2_sbert",
    ascending=False
).head(10)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)




X_e5 = np.vstack(df["e5_for_modeling"])

y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)


score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

df["similarity_1_5e"] = score_1.flatten()

df[df["similarity_1_5e"]>0.3][["clean_job_title", "similarity_1_5e"]].sort_values(
    by="similarity_1_5e",
    ascending=False
).head(10)

In [ ]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_e5
)

df["similarity_2_5e"] = score_2.flatten()

df[df["similarity_2_5e"]>0.3][["clean_job_title", "similarity_2_5e"]].sort_values(
    by="similarity_2_5e",
    ascending=False
).head(10)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

bge = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


def get_title_vector_bge(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = bge(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy() 


df["bge_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_bge
)


X_bge = np.vstack(df["bge_for_modeling"])

y1_vector = get_title_vector_bge(measure_rank_1)
y2_vector = get_title_vector_bge(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bge
)

df["similarity_1_bge"] = score_1.flatten()
# Chain the filtering and sorting together
df[df["similarity_1_bge"] > 0.3][["clean_job_title", "similarity_1_bge"]].sort_values(
    by="similarity_1_bge",
    ascending=False
).head(10)

In [ ]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bge
)

df["similarity_2_bge"] = score_2.flatten()

df[df["similarity_2_bge"] > 0.3][["clean_job_title", "similarity_2_bge"]].sort_values(
    by="similarity_2_bge",
    ascending=False
).head(10)